# Notebook 02 — BRSET In-Domain Mitigation
**BECITHCON 2026 · Fairness under domain shift · Experiment 2 of 4**

Notebook 01 showed defensible subgroup disparities, large **sensitivity** gaps at the
screening threshold and nonzero **calibration** gaps, with image quality the worst axis.
This notebook asks the reviewer's next question: **can post-hoc methods close those gaps,
and what do they cost?** No retraining is required; everything reuses the predictions
notebook 01 already saved.

**Two cheap, deployable levers**
1. **Group-aware calibration** (per-group temperature scaling, fit on validation). Tests
   whether calibration disparities shrink. We contrast it against *global* calibration to
   show that naive recalibration does **not** fix a group gap.
2. **Group-specific thresholds** (per-group operating point, fit on validation). Tests
   whether sensitivity can be equalized, and measures the false-positive-rate cost.

Calibration maps and thresholds are learned on **validation** and applied to **test**, so
nothing is fit on the data we evaluate on. Bootstrap CIs resample test patients and apply
the already-learned transform.

**Deployment caveat we will discuss in the paper:** quality- and camera-specific thresholds
are operationally fine because both are known at inference. Sex-specific thresholds use a
protected attribute in the decision rule, which is ethically loaded; we report it but flag it.


In [ ]:
# ============================== CONFIG ==============================
LABELS_CSV   = "labels_brset.csv"
SPLIT_FILE   = "split.csv"
PREDS_CSV    = "fairness_outputs/brset_val_test_preds.csv"   # saved by notebook 01
OUTPUT_DIR   = "fairness_outputs"

SEED               = 42
TARGET_SENSITIVITY = 0.85
N_BOOTSTRAP        = 1000
ECE_BINS           = 10


In [ ]:
import os, numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
np.random.seed(SEED)
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize_scalar
os.makedirs(OUTPUT_DIR, exist_ok=True)

LABELS = ['diabetic_retinopathy','macular_edema','scar','nevus','amd',
          'vascular_occlusion','hypertensive_retinopathy','drusens','hemorrhage',
          'myopic_fundus','increased_cup_disc']
HEADLINE_LABELS = ['diabetic_retinopathy','macular_edema','amd','drusens','increased_cup_disc']
AXES = {'sex':'sex_group','quality':'quality_group','camera':'camera_group'}


## Section 1 — Reload data and predictions (no inference)

In [ ]:
df = pd.read_csv(LABELS_CSV)
sp = pd.read_csv(SPLIT_FILE)
df = df.merge(sp[['patient_id','split']], on='patient_id', how='left')

df['sex_group']     = df['patient_sex'].map({1:'sex_1', 2:'sex_2'})
df['quality_group'] = df['quality']
df['camera_group']  = df['camera']

preds = pd.read_csv(PREDS_CSV)
data  = df.merge(preds, on='image_id', how='inner')
val   = data[data['split']=='val'].copy()
test  = data[data['split']=='test'].copy()
print("val:", len(val), "test:", len(test))


## Section 2 — Metric helpers (self-contained, same definitions as notebook 01)

In [ ]:
def safe_auc(y,p):
    y=np.asarray(y); p=np.asarray(p)
    return roc_auc_score(y,p) if len(np.unique(y))>1 else np.nan

def ece(y,p,n_bins=ECE_BINS):
    y=np.asarray(y); p=np.asarray(p)
    if len(y)==0: return np.nan
    bins=np.linspace(0,1,n_bins+1); e=0.0
    for lo,hi in zip(bins[:-1],bins[1:]):
        m=(p>lo)&(p<=hi)
        if m.sum()==0: continue
        e+=m.mean()*abs(y[m].mean()-p[m].mean())
    return e

def sens_fpr(y,p,thr):
    y=np.asarray(y); pred=(np.asarray(p)>=thr).astype(int)
    tp=((pred==1)&(y==1)).sum(); fn=((pred==0)&(y==1)).sum()
    fp=((pred==1)&(y==0)).sum(); tn=((pred==0)&(y==0)).sum()
    return (tp/(tp+fn) if tp+fn else np.nan, fp/(fp+tn) if fp+tn else np.nan)

def thr_for_sens(y,p,target):
    y=np.asarray(y); p=np.asarray(p)
    if y.sum()==0: return 0.5
    chosen=np.unique(p).min()
    for t in np.unique(p)[::-1]:
        s,_=sens_fpr(y,p,t)
        if s>=target: chosen=t; break
    return float(chosen)

def clip01(p, eps=1e-6): return np.clip(p, eps, 1-eps)
def to_logit(p): p=clip01(np.asarray(p)); return np.log(p/(1-p))
def from_logit(z): return 1/(1+np.exp(-z))

def fit_temperature(y, p):
    # single scalar T minimizing BCE on (y, sigmoid(logit(p)/T))
    z=to_logit(p); y=np.asarray(y)
    def nll(T):
        q=clip01(from_logit(z/T))
        return -np.mean(y*np.log(q)+(1-y)*np.log(1-q))
    return float(minimize_scalar(nll, bounds=(0.05,20), method='bounded').x)


## Section 3 — Baseline gaps (the "before")

Per-label thresholds fixed on validation, single global threshold, no calibration.


In [ ]:
GLOBAL_THR = {l: thr_for_sens(val[l], val['prob_'+l], TARGET_SENSITIVITY) for l in LABELS}

def gap_for(frame, axis, label, metric, thr_map=None, prob_col=None, per_group_thr=None):
    col=AXES[axis]; pc=prob_col or ('prob_'+label); vals=[]
    for g,gdf in frame.dropna(subset=[col]).groupby(col):
        y=gdf[label].values; p=gdf[pc].values
        if metric=='auc': v=safe_auc(y,p)
        elif metric=='ece': v=ece(y,p)
        else:
            t=(per_group_thr[g] if per_group_thr else thr_map[label])
            s,f=sens_fpr(y,p,t); v=s if metric=='sensitivity' else f
        if not np.isnan(v): vals.append(v)
    return (max(vals)-min(vals)) if len(vals)>=2 else np.nan

before=[]
for axis in AXES:
    for l in HEADLINE_LABELS:
        before.append(dict(axis=axis,label=l,
            sens_gap=gap_for(test,axis,l,'sensitivity',thr_map=GLOBAL_THR),
            fpr_gap =gap_for(test,axis,l,'fpr',       thr_map=GLOBAL_THR),
            ece_gap =gap_for(test,axis,l,'ece')))
before=pd.DataFrame(before)
print("Baseline gaps (before mitigation):")
print(before.round(3).to_string(index=False))


## Section 4 — Mitigation A: calibration (global vs group-aware)

Global temperature is fit on all of validation. Group-aware fits one temperature per group.
We add calibrated probability columns to the test frame and recompute ECE gaps.


In [ ]:
for l in LABELS:
    # global temperature (per label)
    Tg = fit_temperature(val[l], val['prob_'+l])
    test['probG_'+l] = from_logit(to_logit(test['prob_'+l])/Tg)
    # group-aware temperature (per label, per axis) -> store one calibrated col per axis
    for axis,col in AXES.items():
        newcol = f'prob{axis}_'+l
        test[newcol] = test['prob_'+l].values
        for g,gv in val.dropna(subset=[col]).groupby(col):
            T = fit_temperature(gv[l], gv['prob_'+l])
            m = test[col]==g
            test.loc[m, newcol] = from_logit(to_logit(test.loc[m,'prob_'+l])/T)

calib=[]
for axis in AXES:
    for l in HEADLINE_LABELS:
        calib.append(dict(axis=axis,label=l,
            ece_gap_before = gap_for(test,axis,l,'ece'),
            ece_gap_global = gap_for(test,axis,l,'ece',prob_col='probG_'+l),
            ece_gap_groupaware = gap_for(test,axis,l,'ece',prob_col=f'prob{axis}_'+l)))
calib=pd.DataFrame(calib)
print("ECE gap: before vs global-calibration vs group-aware-calibration")
print(calib.round(3).to_string(index=False))


## Section 5 — Mitigation B: group-specific thresholds

For each group we pick the threshold on validation that reaches the target sensitivity within
that group, then apply it to that group on test. This equalizes sensitivity by construction
on validation; we check the residual gap on test and measure the false-positive-rate cost.


In [ ]:
def per_group_thresholds(val, axis, label, target):
    col=AXES[axis]; out={}
    for g,gv in val.dropna(subset=[col]).groupby(col):
        out[g]=thr_for_sens(gv[label], gv['prob_'+label], target)
    return out

thr_mit=[]
PER_GROUP_THR={}
for axis in AXES:
    for l in HEADLINE_LABELS:
        pgt=per_group_thresholds(val,axis,l,TARGET_SENSITIVITY); PER_GROUP_THR[(axis,l)]=pgt
        thr_mit.append(dict(axis=axis,label=l,
            sens_gap_before=gap_for(test,axis,l,'sensitivity',thr_map=GLOBAL_THR),
            sens_gap_after =gap_for(test,axis,l,'sensitivity',per_group_thr=pgt),
            fpr_gap_before =gap_for(test,axis,l,'fpr',thr_map=GLOBAL_THR),
            fpr_gap_after  =gap_for(test,axis,l,'fpr',per_group_thr=pgt)))
thr_mit=pd.DataFrame(thr_mit)
print("Sensitivity and FPR gaps: before (global thr) vs after (group-specific thr)")
print(thr_mit.round(3).to_string(index=False))


## Section 6 — Bootstrap CIs on the post-mitigation gaps

The learned transforms (group temperatures, group thresholds) are held fixed; we resample
**test patients** and recompute the residual gap, giving 95% CIs on the after-mitigation gaps.


In [ ]:
def bootstrap_after(frame, axis, label, kind, B=N_BOOTSTRAP, seed=SEED):
    rng=np.random.default_rng(seed); col=AXES[axis]
    by={pid:idx.values for pid,idx in frame.groupby('patient_id').groups.items()}
    pts=frame['patient_id'].unique(); vals=[]
    pgt=PER_GROUP_THR.get((axis,label))
    for _ in range(B):
        samp=rng.choice(pts,len(pts),replace=True)
        bf=frame.loc[np.concatenate([by[p] for p in samp])]
        per=[]
        for g,gdf in bf.dropna(subset=[col]).groupby(col):
            y=gdf[label].values
            if kind=='ece_groupaware':
                v=ece(y, gdf[f'prob{axis}_'+label].values)
            elif kind=='sens_after':
                s,_=sens_fpr(y, gdf['prob_'+label].values, pgt[g]); v=s
            elif kind=='fpr_after':
                _,f=sens_fpr(y, gdf['prob_'+label].values, pgt[g]); v=f
            if not np.isnan(v): per.append(v)
        if len(per)>=2: vals.append(max(per)-min(per))
    return (np.mean(vals), np.percentile(vals,2.5), np.percentile(vals,97.5)) if vals else (np.nan,)*3

ci=[]
for axis in AXES:
    for l in HEADLINE_LABELS:
        for kind in ['ece_groupaware','sens_after','fpr_after']:
            m,lo,hi=bootstrap_after(test,axis,l,kind)
            ci.append(dict(axis=axis,label=l,kind=kind,gap=m,ci_low=lo,ci_high=hi,
                           excludes_zero=(lo>0)))
ci=pd.DataFrame(ci)
print("Residual gaps that STILL exclude zero after mitigation (remaining unfairness):")
print(ci[ci.excludes_zero].round(3).to_string(index=False))


## Section 7 — Utility cost of the threshold mitigation

Equalizing sensitivity is only worth it if the false-positive cost is acceptable. We report
overall test sensitivity and FPR before and after, plus the worst-group FPR increase.


In [ ]:
rows=[]
for l in HEADLINE_LABELS:
    y=test[l].values; p=test['prob_'+l].values
    s0,f0=sens_fpr(y,p,GLOBAL_THR[l])
    # apply camera-specific thresholds as the deployable example (camera known at inference)
    pgt=PER_GROUP_THR[('camera',l)]; col=AXES['camera']
    pred=np.zeros(len(test),int)
    for g,t in pgt.items():
        m=(test[col]==g).values; pred[m]=(p[m]>=t).astype(int)
    tp=((pred==1)&(y==1)).sum(); fn=((pred==0)&(y==1)).sum()
    fp=((pred==1)&(y==0)).sum(); tn=((pred==0)&(y==0)).sum()
    s1=tp/(tp+fn) if tp+fn else np.nan; f1=fp/(fp+tn) if fp+tn else np.nan
    rows.append(dict(label=l, sens_before=s0, sens_after=s1, fpr_before=f0, fpr_after=f1))
cost=pd.DataFrame(rows)
print("Camera-specific thresholds: overall utility before vs after")
print(cost.round(3).to_string(index=False))


## Section 8 — Figure and export

In [ ]:
# before/after sensitivity gap by axis for DR (the clinical headline + cross-domain bridge)
fig,ax=plt.subplots(figsize=(7,4))
sub=thr_mit[thr_mit.label=='diabetic_retinopathy']
x=np.arange(len(sub)); w=0.35
ax.bar(x-w/2, sub['sens_gap_before'], w, label='global threshold')
ax.bar(x+w/2, sub['sens_gap_after'],  w, label='group-specific threshold')
ax.set_xticks(x); ax.set_xticklabels(sub['axis']); ax.set_ylabel('sensitivity gap')
ax.set_title('DR sensitivity gap before vs after mitigation'); ax.legend()
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR,'figD_mitigation_dr.png'),dpi=200); plt.close()

before.to_csv(os.path.join(OUTPUT_DIR,'mitigation_baseline_gaps.csv'),index=False)
calib.to_csv(os.path.join(OUTPUT_DIR,'mitigation_calibration.csv'),index=False)
thr_mit.to_csv(os.path.join(OUTPUT_DIR,'mitigation_thresholds.csv'),index=False)
ci.to_csv(os.path.join(OUTPUT_DIR,'mitigation_after_ci.csv'),index=False)
cost.to_csv(os.path.join(OUTPUT_DIR,'mitigation_cost.csv'),index=False)
print("Saved figD_mitigation_dr.png and five mitigation CSVs to", OUTPUT_DIR)


## Section 9 — Reading the mitigation results (for the paper)

- **Calibration (Section 4):** if `ece_gap_global` stays close to `ece_gap_before` while
  `ece_gap_groupaware` drops, that is the clean message, *naive recalibration does not fix
  the group gap; group-aware calibration does*. That contrast is the publishable point.
- **Thresholds (Section 5):** `sens_gap_after` should be much smaller than `sens_gap_before`.
  Watch `fpr_gap_after`, if equalizing sensitivity inflates the false-positive gap, say so;
  honest reporting of the trade-off is stronger than pretending mitigation is free.
- **Residual unfairness (Section 6):** any gap that *still* excludes zero after mitigation is
  important, it means post-hoc fixes are not enough and motivates the cross-domain question.
- **Cost (Section 7):** report the overall sensitivity/FPR change so a reader sees the price.
- **Ethics:** present camera- and quality-specific thresholds as deployable; flag
  sex-specific thresholds as using a protected attribute in the decision rule.

**Next:** Notebook 03 repeats the audit on mBRSET (portable camera) to test whether these
in-domain disparities survive the domain shift, then notebook 04 asks whether adaptation
that restores accuracy also preserves fairness.
